# 01 — Data Preprocessing Pipeline

**Project**: From Clinical Jargon to Plain Language — Medical Text Simplification  
**Purpose**: Load PLABA + Cochrane datasets, extract sentence pairs, apply quality filters, split and save.  
**Output**: `data/processed/` (Arrow format), `results/data_stats.csv`  
**Run**: Top-to-bottom on a fresh Colab T4 runtime. Expected time: < 10 minutes.

## 0. Colab Setup

**Run this section ONCE on a fresh runtime. Skip if already done.**

Steps:
1. Run the cell below — it clones the repo and sets the working directory.
2. Upload your PLABA `.json` file when prompted.
3. Cochrane loads from HuggingFace automatically — no upload needed.

In [ ]:
# ── COLAB SETUP ──────────────────────────────────────────────────────────────
from google.colab import userdata
_token = userdata.get("GITHUB_TOKEN")  # Set in Colab Secrets (key icon in sidebar)
GITHUB_URL = f"https://{_token}@github.com/IbrahimHanafy2222/NLP-Project.git"

# ─────────────────────────────────────────────────────────────────────────────

import os, sys, subprocess

# Clone repo
result = subprocess.run(['git', 'clone', GITHUB_URL], capture_output=True, text=True, cwd='/content')
print(result.stdout or result.stderr)

# Auto-detect cloned folder (no hardcoded name needed)
content_dirs = [d for d in os.listdir('/content')
                if os.path.isdir(f'/content/{d}') and d not in ['sample_data', '.config']]
print('Folders in /content:', content_dirs)

# Find the folder that contains src/ — that is our project
project_root = None
for d in content_dirs:
    if os.path.isdir(f'/content/{d}/src'):
        project_root = f'/content/{d}'
        break

if project_root is None:
    raise RuntimeError(
        f'Project folder not found. Folders in /content: {content_dirs}\n'
        'Make sure src/ exists in your repo and the clone succeeded.'
    )

# Set working directory and make src/ importable
os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print('✓ Working directory:', os.getcwd())

# Create data dirs
os.makedirs('data/plaba', exist_ok=True)
os.makedirs('data/cochrane', exist_ok=True)

# Upload PLABA JSON file
print('\n--- Upload your PLABA .json file now ---')
from google.colab import files
uploaded = files.upload()
for fname in uploaded:
    dest = f'data/plaba/{fname}'
    with open(dest, 'wb') as f:
        f.write(uploaded[fname])
    print(f'✓ Saved {fname} → {dest}')

print('\n✓ Colab setup complete. Proceed to Section 1.')

## 1. Setup

In [ ]:
# Clean install (Colab-friendly)
!pip install datasets==2.18.0 textstat spacy==3.7.4 pandas==2.2.1 scikit-learn==1.4.1.post1

# Install model
!python -m spacy download en_core_web_sm

# Restart runtime (important)
import os
os.kill(os.getpid(), 9)

## 2. Configuration

In [ ]:
# Imports and seed setting
import random
import os
import sys
import json
import glob

import numpy as np
import pandas as pd
import spacy
import textstat
from datasets import Dataset, DatasetDict, load_dataset, load_from_disk
from sklearn.model_selection import train_test_split

# Ensure project root is in path (handles both local and Colab)
project_root = os.getcwd()
if 'notebooks' in project_root:
    project_root = os.path.dirname(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data_utils import (
    split_into_sentences,
    align_pairs,
    compute_stats,
    apply_length_ratio_filter,
    apply_fkgl_filter,
    apply_token_count_filter,
)

# Constitution Principle I: fixed seeds everywhere
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print('✓ Imports and seeds set')
print('✓ Project root:', project_root)

In [ ]:
# FilterConfig — all thresholds in one place
FilterConfig = {
    'length_ratio_min': 0.3,
    'length_ratio_max': 3.0,
    'fkgl_diff_min': 1.0,
    'min_source_tokens': 20,
    'max_source_tokens': 200,
    'min_target_tokens': 10,
    'max_target_tokens': 150,
    'train_ratio': 0.80,
    'val_ratio':   0.10,
    'test_ratio':  0.10,
    'random_seed': SEED,
    'spacy_model': 'en_core_web_sm',
}

# Paths (relative to project root)
PLABA_DIR     = os.path.join(project_root, 'data/plaba')
COCHRANE_DIR  = os.path.join(project_root, 'data/cochrane')
PROCESSED_DIR = os.path.join(project_root, 'data/processed')
RESULTS_DIR   = os.path.join(project_root, 'results')

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Load SpaCy model
nlp = spacy.load(FilterConfig['spacy_model'])
if 'sentencizer' not in nlp.pipe_names:
    nlp.add_pipe('sentencizer')

print('✓ FilterConfig ready')
print('✓ SpaCy model loaded')
print('✓ PLABA dir:', PLABA_DIR)
print('✓ Output dir:', PROCESSED_DIR)

## 3. Load Data

In [ ]:
import glob, os, json

# ── CONFIG ─────────────────────────────────────────────
SOURCE_FIELDS = ['abstract', 'abstract_text', 'source', 'text']
TARGET_FIELDS = ['lay_summary', 'plain_language_summary', 'summary', 'target']

# ── SAFE FIELD PICKER ──────────────────────────────────
def _pick_field(record: dict, candidates: list) -> str:
    for key in candidates:
        if isinstance(record, dict) and key in record and record[key]:
            return str(record[key]).strip()
    return ''

# ── FLATTEN + CLEAN HELPERS ────────────────────────────
def is_valid_record(x):
    return isinstance(x, dict) and ("abstract" in x or "adaptations" in x)

def extract_nested_records(data):
    records = []

    if isinstance(data, dict):
        for outer in data.values():
            if isinstance(outer, dict):
                for rec in outer.values():
                    if is_valid_record(rec):
                        records.append(rec)

    elif isinstance(data, list):
        records = [r for r in data if is_valid_record(r)]

    return records

# ── MAIN PIPELINE ──────────────────────────────────────
plaba_pairs = []
json_files = glob.glob(os.path.join(PLABA_DIR, '*.json'))

if not json_files:
    print('⚠ No PLABA JSON files found in', PLABA_DIR)

else:
    for fpath in json_files:
        with open(fpath, encoding='utf-8') as f:
            data = json.load(f)

        records = extract_nested_records(data)

        for rec in records:

            # ── SOURCE (abstract) ──
            src_raw = rec.get("abstract", {})
            if not isinstance(src_raw, dict):
                continue

            src_text = " ".join(
                src_raw[k] for k in sorted(src_raw.keys(), key=int)
                if isinstance(src_raw[k], str)
            )

            # ── TARGET (adaptations) ──
            tgt_raw = rec.get("adaptations", {})
            if isinstance(tgt_raw, dict):
                tgt_raw = next(iter(tgt_raw.values()), {})

            if not isinstance(tgt_raw, dict):
                continue

            tgt_text = " ".join(
                tgt_raw[k] for k in sorted(tgt_raw.keys(), key=int)
                if isinstance(tgt_raw[k], str)
            )

            # ── SKIP EMPTY ──
            if not src_text or not tgt_text:
                continue

            # ── SENTENCE SPLIT ──
            src_sents = split_into_sentences(src_text, nlp)
            tgt_sents = split_into_sentences(tgt_text, nlp)

            # ── ALIGN ──
            plaba_pairs.extend(
                align_pairs(src_sents, tgt_sents, 'plaba')
            )

    print(f'✓ PLABA: {len(plaba_pairs)} sentence pairs from {len(json_files)} file(s)')

In [ ]:
from datasets import load_dataset

cochrane_pairs = []

dataset = load_dataset("GEM/cochrane-simplification", trust_remote_code=True)

for split in dataset:
    for rec in dataset[split]:
        src_text = rec.get("source", "")
        tgt_text = rec.get("target", "")

        if src_text and tgt_text:
            src_sents = split_into_sentences(src_text, nlp)
            tgt_sents = split_into_sentences(tgt_text, nlp)

            cochrane_pairs.extend(
                align_pairs(src_sents, tgt_sents, "cochrane")
            )

print("✓ Cochrane pairs:", len(cochrane_pairs))

In [ ]:
# Combine and record after_extraction stats
df = pd.DataFrame(plaba_pairs + cochrane_pairs)
print(f'Combined: {len(df)} pairs  (PLABA: {len(plaba_pairs)} | Cochrane: {len(cochrane_pairs)})')

stats_records = []
prev_count = 0
stats_records.append(compute_stats(df, 'after_extraction', prev_count))
prev_count = len(df)

## 4. Filter & Clean

In [ ]:
# Exact deduplication
df = df.drop_duplicates(subset=['source', 'target']).reset_index(drop=True)
stats_records.append(compute_stats(df, 'after_dedup', prev_count))
print(f'after_dedup:               {len(df):6d} pairs  (removed {prev_count - len(df)})')
prev_count = len(df)

In [ ]:
# Length ratio filter
df = apply_length_ratio_filter(df, FilterConfig)
stats_records.append(compute_stats(df, 'after_length_ratio_filter', prev_count))
print(f'after_length_ratio_filter: {len(df):6d} pairs  (removed {prev_count - len(df)})')
prev_count = len(df)

In [ ]:
# FKGL difference filter (slowest — computing readability per sentence)
print('Computing FKGL scores...')
df = apply_fkgl_filter(df, FilterConfig)
stats_records.append(compute_stats(df, 'after_fkgl_filter', prev_count))
print(f'after_fkgl_filter:         {len(df):6d} pairs  (removed {prev_count - len(df)})')
prev_count = len(df)

In [ ]:
# Token count filter + final stats
df = apply_token_count_filter(df, FilterConfig)
stats_records.append(compute_stats(df, 'after_token_count_filter', prev_count))
prev_count = len(df)
stats_records.append(compute_stats(df, 'final', prev_count))
print(f'after_token_count_filter:  {len(df):6d} pairs  (final)')

## 5. Split & Save

In [ ]:
# Stratified 80/10/10 split
train_df, temp_df = train_test_split(
    df, test_size=0.20,
    stratify=df['source_dataset'],
    random_state=FilterConfig['random_seed'],
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50,
    stratify=temp_df['source_dataset'],
    random_state=FilterConfig['random_seed'],
)

print(f'Train: {len(train_df):5d} | Val: {len(val_df):5d} | Test: {len(test_df):5d}')
for name, sdf in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f'  {name}: {sdf["source_dataset"].value_counts().to_dict()}')

In [ ]:
# Save to HuggingFace Arrow format
DatasetDict({
    'train': Dataset.from_pandas(train_df.reset_index(drop=True)),
    'val':   Dataset.from_pandas(val_df.reset_index(drop=True)),
    'test':  Dataset.from_pandas(test_df.reset_index(drop=True)),
}).save_to_disk(PROCESSED_DIR)

print(f'✓ Splits saved → {PROCESSED_DIR}/')

## 6. Verify & Report

In [ ]:
# Save statistics report
stats_df = pd.DataFrame(stats_records)
stats_csv = os.path.join(RESULTS_DIR, 'data_stats.csv')
stats_df.to_csv(stats_csv, index=False)
print(f'✓ Statistics saved → {stats_csv}')

In [ ]:
# Print summary
print('\n=== Pipeline Statistics ===')
cols = ['stage', 'total_pairs', 'pairs_removed', 'avg_source_fkgl', 'avg_target_fkgl']
print(stats_df[cols].to_string(index=False))

print('\n=== Split Summary ===')
for name, sdf in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f'  {name:5s}: {len(sdf):5d} | {sdf["source_dataset"].value_counts().to_dict()}')

In [ ]:
# Reproducibility verification — reload and check sizes match
reloaded = load_from_disk(PROCESSED_DIR)
assert len(reloaded['train']) == len(train_df), 'Train size mismatch!'
assert len(reloaded['val'])   == len(val_df),   'Val size mismatch!'
assert len(reloaded['test'])  == len(test_df),  'Test size mismatch!'
print('✓ Reproducibility check passed')
print('  train[0]:', reloaded['train'][0])

In [ ]:
# Contract compliance assertions
reloaded = load_from_disk(PROCESSED_DIR)
for split_name in ['train', 'val', 'test']:
    split = reloaded[split_name]
    assert 'source' in split.column_names
    assert 'target' in split.column_names
    assert 'source_dataset' in split.column_names
    sdf = split.to_pandas()
    assert sdf['source'].notna().all()
    assert sdf['target'].notna().all()
    datasets_present = set(sdf['source_dataset'].unique())
    print(f'✓ {split_name:5s}: {len(split):5d} pairs | datasets: {datasets_present}')

assert len(reloaded['train']) >= 6000, \
    f"Train too small ({len(reloaded['train'])}). Check data/plaba/ and data/cochrane/."

print('\n✓ All checks passed. data/processed/ ready for downstream notebooks.')

In [ ]:
from google.colab import drive

drive.mount('/drive')
import shutil
shutil.copytree('data/processed', '/drive/MyDrive/nlp_project/processed', dirs_exist_ok=True)
shutil.copy('results/data_stats.csv', '/drive/MyDrive/NLP_Project/data_stats.csv')
print('Saved to Drive')